# Run pyCIAM for Inequality Analysis> **Original Reference:** Adapted from> [`run-pyCIAM-slrquantiles.ipynb`](https://gitlab.com/ClimateImpactLab/coastal/projects/pyciam/-/blob/inequality/notebooks/models/run-pyCIAM-slrquantiles.ipynb)> from the [pyCIAM inequality branch](https://gitlab.com/ClimateImpactLab/coastal/projects/pyciam/-/tree/inequality).**Prerequisites:** Run `01_process_slr_inputs.ipynb` first.**Output:** `pyCIAM_outputs_inequality_1000_ssp234_v2.zarr`

In [ ]:
import sys#sys.path.append("../")

In [ ]:
import numpy as npimport pandas as pdimport xarray as xrfrom collections import OrderedDictfrom itertools import productfrom cloudpathlib import AnyPathfrom config import (    # Paths    PATH_PARAMS,    PATH_SLIIDERS,    PATH_SLIIDERS_SEG,    PATH_SLR_INEQUALITY,    PATH_REFA_INEQUALITY,    PATHS_SURGE_LOOKUP,    PATH_OUTPUT_TMP,    PATH_OUTPUT_INEQUALITY,    PATH_OUTPUT_FINAL,    DIR_SCRATCH,    # Parameters    SEG_VAR,    ADM_VAR,    MC_DIM,    SEG_CHUNKSIZE,    REFA_SEG_CHUNKSIZE,    SAMPLE_CHUNKSIZE,    OUTPUT_YEARS,    OUTPUT_SSPS,    OUTPUT_CASES,    N_SAMPLES_TOTAL,    N_WORKERS_MIN,    N_WORKERS_MAX,    # Metadata    AUTHOR,    CONTACT,    HISTORY,    STORAGE_OPTIONS,    # Functions    save_zarr,)

In [ ]:
# === TEST MODE ===# Set to False for production runTEST_MODE = Trueif TEST_MODE:    from config import *    N_SAMPLES_TOTAL = 5    N_WORKERS_MIN = 2    N_WORKERS_MAX = 4    SEG_CHUNKSIZE = 1    SAMPLE_CHUNKSIZE = 5    PATH_SLR_INEQUALITY = DIR_SCRATCH / "test-ar6-tlim-slr.zarr"    PATH_OUTPUT_TMP = DIR_SCRATCH / "test-pyciam-inequality-tmp.zarr"    PATH_OUTPUT_INEQUALITY = DIR_SCRATCH / "test-pyciam-inequality-output.zarr"    PATH_OUTPUT_FINAL = DIR_SCRATCH / "test-pyciam-inequality-final.zarr"    PATH_REFA_INEQUALITY = DIR_SCRATCH / "test-refa-inequality.zarr"    PATH_SLIIDERS_SEG = DIR_SCRATCH / "test-sliiders-seg-inequality.zarr"

In [ ]:
# Import pyCIAM componentsfrom pyCIAM.constants import CASE_DICT, CASES, COSTTYPES, SOLVCASESfrom pyCIAM.io import (    check_finished_zarr_workflow,    create_template_dataarray,    load_ciam_inputs,)from pyCIAM.run import (    calc_all_cases,    get_refA,    optimize_case,)from pyCIAM.utils import (    add_attrs_to_result,    collapse_econ_inputs_to_seg,    subset_econ_inputs,)

## Configuration

In [ ]:
OVERWRITE = FalseDESCRIPTION = "Projected coastal damages from pyCIAM for inequality analysis, using temperature-limit SLR scenarios."# Set up pathsparams_path = AnyPath(PATH_PARAMS)econ_input_path = str(PATH_SLIIDERS)econ_input_path_seg = PATH_SLIIDERS_SEGslr_input_paths = [PATH_SLR_INEQUALITY]slr_names = ["ar6"]refA_path = PATH_REFA_INEQUALITYsurge_input_paths = {k: AnyPath(v) for k, v in PATHS_SURGE_LOOKUP.items()}output_path = PATH_OUTPUT_TMP# Quantiles = sample indices (1 to N_SAMPLES_TOTAL)quantiles = np.arange(1, N_SAMPLES_TOTAL + 1)# Model config (matching original notebook)diaz_inputs = Falseeps = 1  # eps=1 when diaz_config=Falsestorage_options = STORAGE_OPTIONSmodel_kwargs = {}

In [ ]:
# Read model parametersparams = pd.read_json(params_path)["values"]print("Model parameters:")print(f"  Discount rate: {params.dr}")print(f"  Move factor: {params.movefactor}")print(f"  Planning periods: {params.at_start}")

## Setup Dask Cluster

In [ ]:
import osfrom dask_gateway import Gatewayfrom distributed.diagnostics.plugin import PipInstallfrom distributed import waitimg = os.environ.get("JUPYTER_IMAGE", None)gateway = Gateway()cluster = gateway.new_cluster(    idle_timeout=900,    profile="micro",    **(dict(worker_image=img, scheduler_image=img) if img else {}))client = cluster.get_client()pip_installer = PipInstall(    packages=["cloudpathlib==0.13.0", "rhg_compute_tools"],)client.register_worker_plugin(pip_installer)client.run_on_scheduler(pip_installer.install)cluster.adapt(minimum=N_WORKERS_MIN, maximum=N_WORKERS_MAX)cluster

## Step 1: Collapse SLIIDERS to Segment Level

In [ ]:
if OVERWRITE or not econ_input_path_seg.is_dir():    print("Collapsing SLIIDERS to segment level...")    collapse_econ_inputs_to_seg(        econ_input_path,        econ_input_path_seg,        seg_var_subset=None,        output_chunksize=100,        storage_options=storage_options,        seg_var=SEG_VAR,    )    print("Done.")else:    print(f"Segment-level SLIIDERS already exists at {econ_input_path_seg}")

## Step 2: Load Economic Inputs and Define Output Template

In [ ]:
# Load economic inputsciam_in = subset_econ_inputs(    xr.open_zarr(str(econ_input_path), chunks=None, storage_options=storage_options),    SEG_VAR,    seg_var_subset=None,)# Filter to years around output yearsciam_in = ciam_in.sel(year=np.concatenate((    np.arange(2040, 2060),    np.arange(2080, 2100))))print(f"Economic inputs: {len(ciam_in[SEG_VAR])} segment-regions")

In [ ]:
# Load a single segment to get SLR scenario names for templatethis_seg = ciam_in[SEG_VAR][0].item()test_inputs = load_ciam_inputs(    econ_input_path,    slr_input_paths,    params,    [this_seg],    slr_names=slr_names,    seg_var=SEG_VAR,    surge_lookup_store=None,    mc_dim=MC_DIM,    quantiles=quantiles,    storage_options=storage_options,)slr = test_inputs[1].unstack("scen_mc")scenarios = slr.scenarioprint(f"SLR scenarios: {scenarios.values}")

In [ ]:
# Create output templateattr_dict = {    "updated": pd.Timestamp.now(tz="US/Pacific").strftime("%c"),    "planning_period_start_years": params.at_start,    "author": AUTHOR,    "contact": CONTACT,    "description": DESCRIPTION,    "history": HISTORY,}coords = OrderedDict({    "case": CASES,    "costtype": COSTTYPES,    SEG_VAR: ciam_in[SEG_VAR].values,    "scenario": scenarios,    "sample": quantiles,    "year": np.arange(params.model_start, ciam_in.year.max().item() + 1),    **{dim: ciam_in[dim].values for dim in ["ssp", "iam"] if dim in ciam_in.dims},})chunks = {SEG_VAR: 1, "case": len(coords["case"]) - 1}chunks = {k: -1 if k not in chunks else chunks[k] for k in coords}out_ds = create_template_dataarray(coords.keys(), coords, chunks).to_dataset(name="costs")out_ds["npv"] = out_ds.costs.isel(year=0, costtype=0, drop=True).astype("float64")out_ds["optimal_case"] = out_ds.npv.isel(case=0, drop=True).astype("uint8")out_ds.attrs.update(attr_dict)out_ds = add_attrs_to_result(out_ds)print(f"Output template dims: {dict(out_ds.dims)}")

In [ ]:
# Save templateif OVERWRITE or not output_path.is_dir():    out_ds.to_zarr(        str(output_path),        compute=False,        mode="w",        storage_options=storage_options,    )    print(f"Output template saved to {output_path}")

## Step 3: Calculate Reference Adaptation Heights (refA)

In [ ]:
if OVERWRITE or not refA_path.is_dir():    print("Calculating reference adaptation heights...")    segs = np.unique(ciam_in.seg)    seg_grps = [        segs[i : i + REFA_SEG_CHUNKSIZE]        for i in range(0, len(segs), REFA_SEG_CHUNKSIZE)    ]    samps = np.arange(1, N_SAMPLES_TOTAL + 1)    samp_grps = [        samps[i : i + SAMPLE_CHUNKSIZE]        for i in range(0, len(samps), SAMPLE_CHUNKSIZE)    ]    grps = list(product(seg_grps, samp_grps))    print(f"Processing {len(grps)} refA groups...")    refa_futs = client.map(        get_refA,        grps,        output_path=str(refA_path),        econ_input_path=str(econ_input_path_seg),        slr_input_path=slr_input_paths[0],        params=params,        surge_input_path=surge_input_paths["seg"],        mc_dim=MC_DIM,        storage_options=storage_options,        quantiles=quantiles,        diaz_inputs=diaz_inputs,        eps=eps,        **model_kwargs    )    wait(refa_futs)    n_err = sum(1 for f in refa_futs if f.status == 'error')    print(f"refA done. Errors: {n_err}")else:    print(f"refA already exists at {refA_path}")

## Step 4: Run pyCIAM Cost CalculationsCalculate costs for all adaptation cases across all segment-regions and samples.

In [ ]:
# Create segment groups for calc_all_casesgroups = [    ciam_in[SEG_VAR].isel({SEG_VAR: slice(i, i + SEG_CHUNKSIZE)}).values    for i in np.arange(0, len(ciam_in[SEG_VAR]), SEG_CHUNKSIZE)]# Build groups_ser: maps each seg_var value to its group index.# This is needed later by the optimization step to know which# calc_all_cases groups correspond to each segment.groups_ser = (    pd.Series(groups)    .explode()    .reset_index()    .rename(columns={"index": "group_id", 0: SEG_VAR})    .set_index(SEG_VAR)    .group_id)# Create sample groupssamps = np.arange(1, N_SAMPLES_TOTAL + 1)samp_grps = [    samps[i : i + SAMPLE_CHUNKSIZE]    for i in range(0, len(samps), SAMPLE_CHUNKSIZE)]# Cartesian product of segment groups × sample groupsgrps = list(product(groups, samp_grps))print(f"Total calc_all_cases groups: {len(grps)}")

In [ ]:
# Run Stage 1: Calculate costs for all adaptation casesprint("Running pyCIAM cost calculations...")ciam_futs = np.array(    client.map(        calc_all_cases,        grps,        params=params,        econ_input_path=econ_input_path,        slr_input_paths=slr_input_paths,        slr_names=slr_names,        output_path=output_path,        refA_path=refA_path,        surge_input_path=surge_input_paths[SEG_VAR],        seg_var=SEG_VAR,        mc_dim=MC_DIM,        quantiles=quantiles,        storage_options=storage_options,        diaz_inputs=diaz_inputs,        check=False,        **model_kwargs    ))print(f"Submitted {len(ciam_futs)} tasks")

In [ ]:
# Wait and check resultswait(ciam_futs)n_finished = sum(1 for f in ciam_futs if f.status == 'finished')n_errors = sum(1 for f in ciam_futs if f.status == 'error')print(f"Finished: {n_finished}, Errors: {n_errors}")if n_errors > 0:    for f in ciam_futs:        if f.status == 'error':            try: f.result()            except Exception as e: print(f"First error: {e}")            break

## Step 5: Optimize Adaptation CasesSelect the optimal adaptation strategy for each segment.This splits samples into groups of 250 and, for each segment,identifies which calc_all_cases groups contain its seg_ir values,then calls `optimize_case` to select the best adaptation strategy.**This logic is taken directly from the original `run-pyCIAM-slrquantiles.ipynb`.**

In [ ]:
# Split samples into optimization groups# Original uses 4 groups of 250 for 1000 samples.# Adapt for smaller sample counts in test mode.if N_SAMPLES_TOTAL <= 250:    n_opt_groups = 1    samples_per_group = N_SAMPLES_TOTALelse:    n_opt_groups = 4    samples_per_group = N_SAMPLES_TOTAL // n_opt_groupssample_ids = {}for i in range(n_opt_groups):    sample_ids[i] = np.arange(        samples_per_group * i + 1,        samples_per_group * (i + 1) + 1    )print(f"Optimization: {n_opt_groups} groups of {samples_per_group} samples each")for k, v in sample_ids.items():    print(f"  Group {k}: samples {v[0]}..{v[-1]}")

In [ ]:
# Build the optimization task DataFrame.# For each (seg_ir, sample_group) pair, find which calc_all_cases# group_ids contain seg_ir values belonging to the same segment.# Map each seg_ir to its parent segment's list of seg_ir valuesseg_adm_ser = pd.Series(ciam_in[SEG_VAR].values)seg_adm_ser.index = ciam_in.seg.valuesseg_grps = seg_adm_ser.groupby(seg_adm_ser.index).apply(list)# Start building the task tableprecurser_futs = (    seg_adm_ser.to_frame(SEG_VAR)    .join(seg_grps.rename("seg_group")))# Add sample group indices as a column, then explodeprecurser_futs.loc[:, 'samples'] = [np.arange(0, n_opt_groups)] * len(precurser_futs)precurser_futs = (    precurser_futs    .explode('samples')    .set_index([SEG_VAR, 'samples'])    .seg_group.explode()           # explode list of seg_ir per segment    .to_frame()    .join(groups_ser, on="seg_group")  # get group_id for each seg_ir    .groupby([SEG_VAR, 'samples'])    .group_id.apply(set)           # unique group_ids per (seg_ir, sample_group)    .apply(list))print(f"Optimization tasks: {len(precurser_futs)}")

In [ ]:
# Submit optimization futuresprint("Running optimization...")opt_futs = precurser_futs.reset_index(drop=False).apply(    lambda row: client.submit(        optimize_case,        row[SEG_VAR],        *row.group_id,        quantiles=sample_ids[row['samples']],        econ_input_path=econ_input_path,        output_path=str(output_path),        seg_var=SEG_VAR,        eps=eps,        check=False,        storage_options=storage_options,    ),    axis=1,)print(f"Submitted {len(opt_futs)} optimization tasks")

In [ ]:
# Wait for optimization to completewait(opt_futs)n_opt_ok = sum(1 for f in opt_futs if f.status == 'finished')n_opt_err = sum(1 for f in opt_futs if f.status == 'error')print(f"Optimization finished: {n_opt_ok} ok, {n_opt_err} errors")if n_opt_err > 0:    for f in opt_futs:        if f.status == 'error':            try: f.result()            except Exception as e: print(f"First error: {e}")            break

## Step 6: Extract Final OutputFilter to required cases, years, SSPs, then aggregate from seg_ir to gadmid.

In [ ]:
# Save intermediate filtered outputprint("Extracting filtered output...")t = xr.open_zarr(str(output_path))t = t.sel(    case=OUTPUT_CASES,    ssp=OUTPUT_SSPS,    year=OUTPUT_YEARS,)[["costs"]]for v in t.data_vars:    t[v].encoding.clear()for k, v in t.coords.items():    if v.dtype == object:        t[k] = v.astype("unicode")# Save intermediatePATH_INTERMEDIATE = DIR_SCRATCH / "pyciam-inequality-intermediate.zarr"t.to_zarr(str(PATH_INTERMEDIATE), storage_options=storage_options, mode='w')print(f"Intermediate saved to {PATH_INTERMEDIATE}")

In [ ]:
# Aggregate from seg_ir to gadmidprint("Aggregating to gadmid level...")this_chunksize = 2out = xr.open_zarr(    str(PATH_INTERMEDIATE),    chunks={"case": -1, SEG_VAR: this_chunksize},)out["costs"] = (    out.costs.groupby(ciam_in[ADM_VAR]).sum().chunk({ADM_VAR: this_chunksize})).persist()out = out.drop(SEG_VAR).unify_chunks()for v in out.data_vars:    out[v].encoding.clear()for k, v in out.coords.items():    if v.dtype == object:        out[k] = v.astype("unicode")out = out.persist()print(f"Aggregated output: {dict(out.dims)}")

In [ ]:
# Save final outputprint(f"Saving to {PATH_OUTPUT_FINAL}...")out.to_zarr(str(PATH_OUTPUT_FINAL), storage_options=storage_options, mode="w")print("Done!")

In [ ]:
# Verify outputprint("Verifying output...")verify = xr.open_zarr(str(PATH_OUTPUT_FINAL))print(verify)assert (    verify.sel(case='optimalfixed', drop=True)    .sum(dim='costtype')    .costs.notnull()    .all())print("All values present!")

In [ ]:
# Cleanupclient.close()cluster.close()